# Examen de Recuperación de Información — NFCorpus

**Estudiante:** Jhonn Saeteros  
**Asignatura:** ICCD753 — Recuperación de Información  
**Institución:** Escuela Politécnica Nacional  
**Docente:** Ing. Iván Carrera

## Enunciado

# Examen ICCD753 Recuperación de Información

## 1. Objetivo

Desarrollar un sistema de recuperación de información sobre un corpus asignado. El sistema deberá indexar los documentos, recuperar y ordenar resultados mediante TF-IDF con similitud coseno y BM25, y exponer la búsqueda como un servicio web publicado en internet.

Se evaluará el servicio enviando consultas desde un cliente (p.ej. Postman) y contrastando los resultados con los juicios de relevancia (*qrels*) del corpus.

## 2. Corpus asignado

Cada estudiante recibirá un conjunto de datos. Todos contienen documentos, consultas y qrels. Se utilizarán los identificadores originales de los documentos para que los resultados puedan evaluarse.

| Corpus | Documentos y consultas | Qrels |
| --- | --- | --- |
| NFCorpus | [BeIR/nfcorpus](https://huggingface.co/datasets/BeIR/nfcorpus) | [BeIR/nfcorpus-qrels](https://huggingface.co/datasets/BeIR/nfcorpus-qrels) |
| SciFact | [BeIR/scifact](https://huggingface.co/datasets/BeIR/scifact) | [BeIR/scifact-qrels](https://huggingface.co/datasets/BeIR/scifact-qrels) |

## 3. Requisitos del sistema

### 3.1. Preparación e indexación

El sistema deberá:

1. Cargar e indexar los documentos del corpus asignado.
2. Aplicar y documentar las decisiones de preprocesamiento: tokenización, normalización, eliminación de stopwords y stemming o lematización.
3. Construir un vocabulario y un índice invertido que permita recuperar documentos.
4. Conservar la relación entre cada documento indexado y su identificador original.

La unidad de evaluación será el documento original, porque los qrels relacionan consultas con identificadores de documentos. No se debe dividir el texto en fragmentos.

### 3.2. Modelos de recuperación

Para cualquier consulta de texto, el sistema deberá producir dos rankings independientes:

- TF-IDF con similitud coseno.
- BM25.

Cada ranking deberá ordenar los documentos por puntuación de relevancia y permitir solicitar los primeros `k` resultados.

### 3.3. Servicio web

El sistema deberá estar publicado en internet y ofrecer, como mínimo, los siguientes endpoints:

**Comprobación de disponibilidad**

```http
GET /health
```

**Búsqueda**

```http
POST /search
Content-Type: application/json
```

Ejemplo de solicitud:

```json
{
  "query": "texto de la consulta",
  "model": "bm25",
  "top_k": 10
}
```

El campo `model` deberá aceptar `bm25` y `tfidf`. Una respuesta tendrá esta estructura:

```json
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "identificador_original",
      "score": 8.42,
      "title": "Título, si está disponible",
      "snippet": "Fragmento del documento recuperado"
    }
  ]
}
```

Los resultados deberán estar ordenados de mayor a menor puntuación, no contener documentos duplicados y respetar el valor solicitado de `top_k`. El servicio debe aceptar consultas que el estudiante no conozca de antemano.

## 4. Evaluación de la recuperación

Cada estudiante debe evaluar ambos modelos con consultas de la partición `test` y sus qrels. Presentará, como mínimo:

- Precision@10.
- Recall@10.
- NDCG@10.
- Una comparación de los resultados de TF-IDF y BM25.

En el Jupyter Notebook se deberá indicar qué consultas se utilizaron y permitir reproducir los cálculos. La evaluación del docente podrá utilizar otras consultas del mismo conjunto.

## 5. Prueba del docente

El docente accederá a la URL entregada y realizará solicitudes desde un cliente de web service para comprobar:

1. La disponibilidad del servicio.
2. El funcionamiento de TF-IDF y BM25.
3. El formato, orden y validez de los resultados.
4. La correspondencia de los `document_id` con el corpus y los qrels.
5. La respuesta ante consultas no incluidas en los ejemplos de entrega.

## 6. Entregables

Cada estudiante entregará:

1. La URL pública del servicio web, accesible durante el período de revisión.
2. El código en Jupyter Notebook con los resultados de la ejecución de cada bloque de código, incluyendo los resultados de las métricas de evaluación de los modelos de recuperación.

Se evaluará que ambos modelos funcionen correctamente y que cada estudiante interprete sus resultados.

## 1. Carga de NFCorpus

Se utilizan los documentos completos de NFCorpus, sus consultas y los qrels de
`test`. Cada documento se representa mediante su título y contenido, conservando
el identificador original. El título aporta el tema y el contenido sus detalles.

Dependencias: `python -m pip install -r requirements-dev.txt`. Ejecutar desde
la carpeta del proyecto con ese entorno como kernel.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'retrieval.py').exists())
sys.path.insert(0, str(ROOT))
from preprocessing import preprocess
from retrieval import SearchIndex
from evaluation import load_dataset, evaluate

ARTIFACTS = ROOT / 'artifacts'
ARTIFACTS.mkdir(exist_ok=True)
documents, queries, qrels = load_dataset()

display(pd.DataFrame([{
    'Documentos': len(documents), 'Consultas': len(queries),
    'Consultas test': len(qrels), 'Juicios test': sum(map(len, qrels.values()))}]))
display(pd.DataFrame(documents[:1])[['_id', 'title', 'text']])
display(pd.DataFrame([{'query_id': qid, 'query': queries[qid]}
    for qid in sorted(qrels)[:3]]))

,Documentos,Consultas,Consultas test,Juicios test
0,3633,3237,323,12334


,_id,title,text
0,MED-10,Statin Use and Breast Cancer Survival: A Natio...,"Recent studies have suggested that statins, an..."


,query_id,query
0,PLAIN-1008,deafness
1,PLAIN-1018,DHA
2,PLAIN-102,Stopping Heart Disease in Childhood


## 2. Preprocesamiento

Se aplica el mismo procesamiento a documentos y consultas:

- Normalización Unicode NFKC y conversión a minúsculas.
- Tokenización conservando números, decimales, letras griegas y símbolos relevantes.
- Eliminación de stopwords inglesas, conservando negaciones como `not` y `without`.
- Stemming Snowball en inglés para agrupar variantes morfológicas sin diccionarios externos.

Los signos no contemplados por el tokenizador actúan como separadores. El texto
original se conserva para mostrar títulos y fragmentos en los resultados.

In [2]:
examples = [documents[0]['title'], queries['PLAIN-3332']]
display(pd.DataFrame([{'Texto original': text, 'Tokens procesados': preprocess(text)}
    for text in examples]))

,Texto original,Tokens procesados
0,Statin Use and Breast Cancer Survival: A Natio...,"[statin, use, breast, cancer, surviv, nationwi..."
1,Alcohol Risks vs. Benefits,"[alcohol, risk, vs, benefit]"


## 3. Vocabulario e índice invertido

`vocabulary` asigna una columna a cada término. `postings[t]` almacena pares
`(posición del documento, frecuencia)` y `doc_ids` relaciona las posiciones con
los identificadores originales. La unión de postings determina los candidatos.

Se calculan TF, DF, longitud de los documentos y longitud media (`avgdl`).
Las frecuencias se almacenan en una matriz dispersa.

In [3]:
index = SearchIndex(documents)
index.save(ARTIFACTS / 'index')
assert index.doc_ids == [doc['_id'] for doc in documents]

display(pd.DataFrame([{'Documentos': index.N, 'Términos': len(index.vocabulary),
    'avgdl': index.avgdl}]))
display(pd.DataFrame([{'Término': term, 'Columna': col, 'DF': int(index.df[col])}
    for term, col in list(index.vocabulary.items())[:5]]))
term = preprocess('cholesterol')[0]
display(pd.DataFrame([{'Término': term, 'document_id': index.doc_ids[pos], 'TF': int(tf)}
    for pos, tf in index.postings[term][:5]]))

,Documentos,Términos,avgdl
0,3633,22797,166.370493


,Término,Columna,DF
0,%,0,1580
1,+,1,312
2,-0,2,1
3,-0.0,3,1
4,-0.0.98,4,1


,Término,document_id,TF
0,cholesterol,MED-711,4
1,cholesterol,MED-712,1
2,cholesterol,MED-726,3
3,cholesterol,MED-751,7
4,cholesterol,MED-754,4


## 4. Modelos de recuperación

### TF-IDF con similitud coseno

$$TF(t,d)=1+\ln f(t,d),\qquad IDF(t)=1+\ln\frac{N+1}{df(t)+1}$$

$$w_{t,d}=TF(t,d)IDF(t),\qquad
\cos(q,d)=\frac{\sum_t w_{t,q}w_{t,d}}{\|q\|_2\|d\|_2}.$$

La consulta utiliza el mismo vocabulario e IDF que los documentos. Los términos
ausentes tienen peso cero y los desconocidos se ignoran. Se evitan divisiones
con denominador cero.

### BM25

$$IDF(t)=\ln\left(1+\frac{N-df(t)+0.5}{df(t)+0.5}\right)$$

$$BM25(q,d)=\sum_{t\in unique(q)}IDF(t)
\frac{f(t,d)(k_1+1)}{f(t,d)+k_1(1-b+b|d|/avgdl)}.$$

Se utilizan `k1=1.2` y `b=0.75`, sin ajuste con los qrels de prueba. La frecuencia
se satura y se normaliza según la longitud del documento.

Cada modelo produce un ranking independiente, ordenado por puntuación descendente.
`top_k` controla la cantidad de resultados. Los empates se resuelven por ID original.

In [4]:
query_id = 'PLAIN-3332'
query = queries[query_id]
top_k = 5
print(query_id, '—', query)
for model in ['tfidf', 'bm25']:
    print(model)
    display(pd.DataFrame(index.search(query, model, top_k)))

PLAIN-3332 — Alcohol Risks vs. Benefits
tfidf


,rank,document_id,score,title,snippet
0,1,MED-4994,0.191145,Who benefits most from the cardioprotective pr...,BACKGROUND: The cardioprotective properties of...
1,2,MED-3696,0.188357,Alcohol consumption and breast cancer risk in ...,The authors assessed the association between m...
2,3,MED-3762,0.186291,Moderate alcohol consumption during adult life...,Context Multiple studies have linked alcohol c...
3,4,MED-3768,0.179868,The positive and negative health effects of al...,"In this paper, the negative and the positive e..."
4,5,MED-3766,0.174148,Epidemiology and pathophysiology of alcohol an...,AIMS: To update epidemiological data on alcoho...


bm25


,rank,document_id,score,title,snippet
0,1,MED-1680,11.241494,Effect of potentially modifiable risk factors ...,BACKGROUND: Although more than 80% of the glob...
1,2,MED-3696,10.284088,Alcohol consumption and breast cancer risk in ...,The authors assessed the association between m...
2,3,MED-4994,9.837812,Who benefits most from the cardioprotective pr...,BACKGROUND: The cardioprotective properties of...
3,4,MED-3763,9.440966,A single sip of a strong alcoholic beverage ca...,The aim of this study was to explore oral expo...
4,5,MED-1678,9.201869,Combined effect of low-risk dietary and lifest...,BACKGROUND: Limited data are available on the ...


## 5. Evaluación y comparación

Se evalúan todas las **323 consultas de `qrels/test.tsv`**. Sus identificadores y
resultados individuales quedan en `artifacts/metrics_per_query.csv`.
Los juicios se utilizan después de recuperar, sin intervenir en el ranking.

$$P@10=\frac{\#\text{ relevantes en top 10}}{10},\qquad
R@10=\frac{\#\text{ relevantes en top 10}}{\#\text{ relevantes de la consulta}}$$

$$DCG@10=\sum_{i=1}^{10}\frac{2^{rel_i}-1}{\log_2(i+1)},\qquad
NDCG@10=\frac{DCG@10}{IDCG@10}.$$

Para Precision y Recall se considera relevante un grado mayor que cero.
NDCG conserva los grados de los qrels; IDCG utiliza el orden ideal de todos los
juicios de la consulta. Un documento sin juicio recibe grado cero en el cálculo.
Si no hay relevantes, Recall es cero; si IDCG es cero, NDCG es cero.
Se presentan los promedios macro por consulta.

In [5]:
per_query, summary, runs = evaluate(index, queries, qrels, 10)
per_query.to_csv(ARTIFACTS / 'metrics_per_query.csv', index=False)
summary.to_csv(ARTIFACTS / 'metrics_summary.csv')

display(summary.rename_axis('Modelo').rename(index={'tfidf': 'TF-IDF', 'bm25': 'BM25'}).style.format('{:.6f}'))
display(per_query.head(6))

tf, bm = summary.loc['tfidf'], summary.loc['bm25']
display(Markdown(
    f"BM25 obtuvo Precision@10 de **{bm['Precision@10']:.6f}** y NDCG@10 de "
    f"**{bm['NDCG@10']:.6f}**. TF-IDF obtuvo **{tf['Precision@10']:.6f}** y "
    f"**{tf['NDCG@10']:.6f}**, respectivamente. TF-IDF presentó un Recall@10 "
    f"ligeramente mayor: **{tf['Recall@10']:.6f}**, frente a **{bm['Recall@10']:.6f}**."
))

,Precision@10,Recall@10,NDCG@10
Modelo,,,
TF-IDF,0.232508,0.153092,0.323104
BM25,0.238080,0.152893,0.328409


,model,query_id,Precision@10,Recall@10,NDCG@10
0,tfidf,PLAIN-1008,0.0,0.000000,0.000000
1,tfidf,PLAIN-1018,0.6,0.100000,0.636474
2,tfidf,PLAIN-102,0.0,0.000000,0.000000
3,tfidf,PLAIN-1028,0.0,0.000000,0.000000
4,tfidf,PLAIN-1039,0.2,1.000000,1.000000
5,tfidf,PLAIN-1050,0.1,0.010638,0.063621


BM25 obtuvo Precision@10 de **0.238080** y NDCG@10 de **0.328409**. TF-IDF obtuvo **0.232508** y **0.323104**, respectivamente. TF-IDF presentó un Recall@10 ligeramente mayor: **0.153092**, frente a **0.152893**.

### Interpretación

En `PLAIN-3332` (*Alcohol Risks vs. Benefits*), TF-IDF coloca primero `MED-4994`
(grado 2), mientras BM25 lo sitúa tercero. BM25 favorece `MED-1680`, sin juicio
positivo para esta consulta, por las frecuencias de `alcohol`, `risk` y `vs`.

En `PLAIN-3342` (*Is Coconut Milk Good For You?*), BM25 sitúa primero `MED-5012`
y segundo `MED-5014`, ambos con grado 2. TF-IDF coloca primero `MED-5015`, un
documento corto sobre agua de coco, sin juicio positivo para esta consulta.
Su coincidencia con `coconut` y su norma favorecen el coseno.

La normalización y el tratamiento de la frecuencia explican estas diferencias.
Ambos modelos dependen de coincidencias léxicas y no resuelven automáticamente
los sinónimos o las relaciones entre conceptos.

In [6]:
for qid in ['PLAIN-3332', 'PLAIN-3342']:
    print(qid, '—', queries[qid])
    rows = []
    for model in ['tfidf', 'bm25']:
        for result in runs[model][qid][:3]:
            did = result['document_id']
            rows.append({'Modelo': model, 'Posición': result['rank'], 'document_id': did,
                'Grado': qrels[qid].get(did, 0),
                'Longitud': int(index.doc_lengths[index.id_to_position[did]]),
                'Título': result['title']})
    display(pd.DataFrame(rows))

PLAIN-3332 — Alcohol Risks vs. Benefits


,Modelo,Posición,document_id,Grado,Longitud,Título
0,tfidf,1,MED-4994,2,137,Who benefits most from the cardioprotective pr...
1,tfidf,2,MED-3696,1,175,Alcohol consumption and breast cancer risk in ...
2,tfidf,3,MED-3762,1,180,Moderate alcohol consumption during adult life...
3,bm25,1,MED-1680,0,273,Effect of potentially modifiable risk factors ...
4,bm25,2,MED-3696,1,175,Alcohol consumption and breast cancer risk in ...
5,bm25,3,MED-4994,2,137,Who benefits most from the cardioprotective pr...


PLAIN-3342 — Is Coconut Milk Good For You?


,Modelo,Posición,document_id,Grado,Longitud,Título
0,tfidf,1,MED-5015,0,50,The intravenous use of coconut water.
1,tfidf,2,MED-5012,2,250,The cholesterol-lowering effect of coconut fla...
2,tfidf,3,MED-5016,0,162,"Effects of dietary coconut oil, butter and saf..."
3,bm25,1,MED-5012,2,250,The cholesterol-lowering effect of coconut fla...
4,bm25,2,MED-5014,2,205,Dietary intake and the risk of coronary heart ...
5,bm25,3,MED-5015,0,50,The intravenous use of coconut water.


## 6. Servicio web y URL pública

La API carga el índice al iniciar y utiliza el mismo código de recuperación que
el notebook. Expone `GET /health` y `POST /search` para los modelos `tfidf` y `bm25`.
Cada resultado contiene `rank`, `document_id`, `score`, `title` y `snippet`.

`SERVICE_URL` corresponde a la dirección pública que se entregará. Mientras esté
vacía, la celda comprueba la API local y señala que la publicación está pendiente.

In [7]:
import httpx

SERVICE_URL = ''  # URL pública del servicio

if SERVICE_URL:
    client = httpx.Client(base_url=SERVICE_URL, timeout=180)
    print('URL pública:', SERVICE_URL)
else:
    from fastapi.testclient import TestClient
    from app import app
    client = TestClient(app)
    print('URL pública: pendiente. Verificación de la API local.')

with client:
    health = client.get('/health')
    assert health.status_code == 200
    print('GET /health:', health.status_code, health.json())
    for model in ['tfidf', 'bm25']:
        request = {'query': 'vitamin B12 deficiency', 'model': model, 'top_k': 3}
        response = client.post('/search', json=request)
        assert response.status_code == 200
        results = response.json()['results']
        assert results == index.search(request['query'], model, request['top_k'])
        assert len({r['document_id'] for r in results}) == len(results)
        assert [r['score'] for r in results] == sorted([r['score'] for r in results], reverse=True)
        print('POST /search:', model, response.status_code)
        display(response.json())

C:\Users\Jhonn22\OneDrive - Escuela Politécnica Nacional\Escritorio\!Jhonn Trabajos!\RECUPERACION DE LA INFORMACION\examen1bim\examen1bimJhonnSaeteros\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


URL pública: pendiente. Verificación de la API local.


GET /health: 200 {'status': 'ok', 'documents': 3633, 'models': ['tfidf', 'bm25']}
POST /search: tfidf 200


{'model': 'tfidf',
 'results': [{'rank': 1,
   'document_id': 'MED-5132',
   'score': 0.6083529307598773,
   'title': 'Role of vitamin B12 in depressive disorder--a case report.',
   'snippet': 'Vitamin B12 deficiency anemia may have psychiatric manifestations preceding the hematological symptoms. Although a variety of symptoms are described, there are only sparse data on the role of vitamin B12 in depression. We report a case of vitamin B12 deficiency presenting with recurrent episodes of '},
  {'rank': 2,
   'document_id': 'MED-4685',
   'score': 0.5203137472640318,
   'title': 'Maternal and neonatal vitamin B12 deficiency detected through expanded newborn screening--United States, 2003-2007.',
   'snippet': 'The incidence of neonatal vitamin B12 (cobalamin) deficiency because of maternal deficiency was determined by surveying state newborn screening programs. Thirty-two infants with nutritional vitamin B12 deficiency were identified (0.88/100,000 newborns). Pregnant women should be 

POST /search: bm25 200


{'model': 'bm25',
 'results': [{'rank': 1,
   'document_id': 'MED-4685',
   'score': 21.902067258096558,
   'title': 'Maternal and neonatal vitamin B12 deficiency detected through expanded newborn screening--United States, 2003-2007.',
   'snippet': 'The incidence of neonatal vitamin B12 (cobalamin) deficiency because of maternal deficiency was determined by surveying state newborn screening programs. Thirty-two infants with nutritional vitamin B12 deficiency were identified (0.88/100,000 newborns). Pregnant women should be assessed for their ri'},
  {'rank': 2,
   'document_id': 'MED-5132',
   'score': 21.62298768401813,
   'title': 'Role of vitamin B12 in depressive disorder--a case report.',
   'snippet': 'Vitamin B12 deficiency anemia may have psychiatric manifestations preceding the hematological symptoms. Although a variety of symptoms are described, there are only sparse data on the role of vitamin B12 in depression. We report a case of vitamin B12 deficiency presenting with rec